# Planning Lab

In [2]:
!pip install unified-planning==1.1.0
!pip install up_fast_downward==0.4.1

In [3]:
!pip install matplotlib==3.7.1  # for visualisation in this notebook

In [1]:
from unified_planning.shortcuts import *

import unified_planning as up

up.shortcuts.get_environment().credits_stream = None

/home/axegl999/Documents/Master/tddc17/venv/lib/python3.12/site-packages/up_fast_downward/fast_downward.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
%pip install setuptools
%pip install "setuptools<81"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Part (a): Model the Task

The code below models a simpler version of the Household Robot domain with two rooms and one open door between them. The robot is in room A initially and needs to move to room B. You can use it as a starting point for your own solution to the full Household Robot task.

In [4]:

def get_planning_task():
    # Declare user types.
    Room = UserType("Room")
    Door = UserType("Door")
    Key = UserType("key")

    # Declare predicates.
    robot_in = up.model.Fluent("robot_in", BoolType(), r=Room)
    connected = up.model.Fluent("connected", BoolType(), r1=Room, d=Door, r2=Room)
    room_has_door = up.model.Fluent("room_has_door", BoolType(), r=Room, d=Door)
    key_at = up.model.Fluent("key_at", BoolType(), k=Key, r=Room)
    robot_has = up.model.Fluent("robot_has", BoolType(), k=Key)
    key_fits = up.model.Fluent("key_fits", BoolType(), k=Key, d=Door)
    locked = up.model.Fluent("locked", BoolType(), d=Door)

    # Add (typed) objects to problem.
    problem = up.model.Problem("household")

    def get_key(key):
        return up.model.Object(f"key{key}", Key)
    
    def get_room(room):
        return up.model.Object(f"room{room}", Room)

    def get_door(door):
        return up.model.Object(f"door{door}", Door)

    livingroom = get_room("LIVING")
    corridor = get_room("CORRIDOR")
    bathroom=get_room("BATHROOM")
    lobby=get_room("LOBBY")
    kitchen=get_room("KITCHEN")
    rooms = [livingroom, corridor, bathroom, lobby, kitchen]

    doorK = get_door("K")
    doorL = get_door("L")
    doorC = get_door("C")
    doorB = get_door("B")
    doorF = get_door("F")
    doors = [doorK, doorL, doorC, doorB, doorF]

    keyK = get_key("K")
    keyL = get_key("L")
    keyB = get_key("B")
    keyC = get_key("C")
    keyF = get_key("F")
    keyAll = get_key("All")
    keys = [keyK, keyL, keyB, keyC, keyF, keyAll]

    problem.add_objects(rooms)
    problem.add_objects(doors)
    problem.add_objects(keys)

    connections = [
            (kitchen, doorK, livingroom),
            (livingroom, doorL, corridor),
            (corridor, doorB, bathroom),
            (corridor, doorC, lobby)
        ]

    rooms_and_doors = []
    for room1, door, room2 in connections:
        rooms_and_doors.append((room1, door))
        rooms_and_doors.append((room2, door))
    rooms_and_doors.append((lobby, doorF))

    key_positions = [
        (keyK, livingroom),
        (keyL, livingroom),
        (keyAll, kitchen),
        (keyB, corridor),
        (keyC, bathroom),
        (keyF, bathroom)
    ]

    fitting_pairs = [
            (keyK, doorK),
            (keyL, doorL),
            (keyB, doorB),
            (keyC, doorC),
            (keyF, doorF),
        ]

    # Specify the initial state.
    problem.add_fluent(robot_in, default_initial_value=False)
    problem.add_fluent(connected, default_initial_value=False)
    problem.add_fluent(room_has_door, default_initial_value=False)
    problem.add_fluent(key_at, default_initial_value=False)
    problem.add_fluent(robot_has, default_initial_value=False)
    problem.add_fluent(key_fits, default_initial_value=False)
    problem.add_fluent(locked, default_initial_value=False)
    problem.set_initial_value(robot_in(livingroom), True)

    for room1, door, room2 in connections:
        problem.set_initial_value(connected(room1, door, room2), True)
        problem.set_initial_value(connected(room2, door, room1), True)

    for room, door in rooms_and_doors:
        problem.set_initial_value(room_has_door(room, door), True)

    for key, room in key_positions:
            problem.set_initial_value(key_at(key,room), True)
    
    for key, door in fitting_pairs:
        problem.set_initial_value(key_fits(key, door), True)
    
    for door in doors:
        problem.set_initial_value(key_fits(keyAll, door), True)
        problem.set_initial_value(locked(door), True)

    # Action: move
    move = up.model.InstantaneousAction("move", room1=Room, door=Door, room2=Room)
    room1 = move.parameter("room1")
    door = move.parameter("door")
    room2 = move.parameter("room2")
    move.add_precondition(robot_in(room1))
    move.add_precondition(connected(room1, door, room2))
    move.add_precondition(Not(locked(door)))
    move.add_effect(robot_in(room1), False)
    move.add_effect(robot_in(room2), True)
    problem.add_action(move)

    # Action: pick up keys
    pick_up = up.model.InstantaneousAction("pick_up", key=Key, room=Room)
    key = pick_up.parameter("key")
    room = pick_up.parameter("room")
    pick_up.add_precondition(robot_in(room))
    pick_up.add_precondition(key_at(key, room))
    pick_up.add_effect(robot_has(key), True)
    pick_up.add_effect(key_at(key,room), False)
    problem.add_action(pick_up)

    # Action: open
    open_door = up.model.InstantaneousAction("open_door", room=Room, door=Door, key=Key)
    room = open_door.parameter("room")
    door = open_door.parameter("door")
    key = open_door.parameter("key") 
    open_door.add_precondition(robot_in(room))
    open_door.add_precondition(room_has_door(room, door))
    open_door.add_precondition(robot_has(key))
    open_door.add_precondition(key_fits(key,door))
    open_door.add_precondition(locked(door))
    open_door.add_effect(locked(door), False)
    problem.add_action(open_door)

    # Specify the goal.
    problem.add_goal(Not(locked(doorF)))

    # We want to minimize the plan cost.
    problem.add_quality_metric(MinimizeActionCosts({}, default=Int(1)))
    return problem

problem = get_planning_task()


## Part (b): Find a (possibly suboptimal) plan

Solve the task with greedy best-first search using the FF heuristic. The example code below uses the h^add heuristic. You need to inspect the output to stdout to see the heuristic value of the initial state.

In [10]:
params = {
    # switched add to ff (heuristics), to extract the actuall relaxed plan
    # add will double count everything, ff will not. 
    "fast_downward_search_config": "eager_greedy([add()])"
}

with OneshotPlanner(name="fast-downward", params=params) as planner:
    result = planner.solve(problem, output_stream=sys.stdout)
    if result.status == up.engines.PlanGenerationResultStatus.SOLVED_SATISFICING:
        print("Found a plan of length:", len(result.plan.actions))
        print(result.plan)
        with PlanValidator() as validator:
            val_result = validator.validate(problem, result.plan)
            print("Plan cost:", val_result.metric_evaluations)
    else:
        print("No plan found.")

INFO     planner time limit: None
INFO     planner memory limit: None

INFO     Running translator.
INFO     translator stdin: None
INFO     translator time limit: None
INFO     translator memory limit: None
INFO     translator command line string: /home/axegl999/Documents/Master/tddc17/venv/bin/python /home/axegl999/Documents/Master/tddc17/venv/lib/python3.12/site-packages/up_fast_downward/downward/builds/release/bin/translate/translate.py /tmp/tmpvs9asuye/domain.pddl /tmp/tmpvs9asuye/problem.pddl --sas-file output.sas
Parsing...
Parsing: [0.000s CPU, 0.003s wall-clock]
Normalizing task... [0.000s CPU, 0.000s wall-clock]
Instantiating...
Generating Datalog program... [0.000s CPU, 0.001s wall-clock]
Normalizing Datalog program...
Trivial rules: Converted to facts.
Normalizing Datalog program: [0.000s CPU, 0.005s wall-clock]
Preparing model... [0.000s CPU, 0.001s wall-clock]
Generated 16 rules.
Computing model... [0.000s CPU, 0.002s wall-clock]
130 relevant atoms
76 auxiliary atoms
206 

## Part (c): Find an optimal plan

Solve the task with A* using the `iPDB` heuristic.

In [12]:
params = {
    # switched add to ff (heuristics), to extract the actuall relaxed plan
    # add will double count everything, ff will not. 
    "fast_downward_search_config": "astar(ipdb())"
}

with OneshotPlanner(name="fast-downward", params=params) as planner:
    result = planner.solve(problem, output_stream=sys.stdout)
    solved = {
        up.engines.PlanGenerationResultStatus.SOLVED_SATISFICING,
        up.engines.PlanGenerationResultStatus.SOLVED_OPTIMALLY
    }


    if result.status in solved:
        print("Status: ", result.status)
        print("Found a plan of length:", len(result.plan.actions))
        print(result.plan)
        with PlanValidator() as validator:
            val_result = validator.validate(problem, result.plan)
            print("Plan cost:", val_result.metric_evaluations)
    else:
        print("No plan found. Status: ", result.status)

INFO     planner time limit: None
INFO     planner memory limit: None

INFO     Running translator.
INFO     translator stdin: None
INFO     translator time limit: None
INFO     translator memory limit: None
INFO     translator command line string: /home/axegl999/Documents/Master/tddc17/venv/bin/python /home/axegl999/Documents/Master/tddc17/venv/lib/python3.12/site-packages/up_fast_downward/downward/builds/release/bin/translate/translate.py /tmp/tmptmpvyitl/domain.pddl /tmp/tmptmpvyitl/problem.pddl --sas-file output.sas
Parsing...
Parsing: [0.000s CPU, 0.003s wall-clock]
Normalizing task... [0.000s CPU, 0.000s wall-clock]
Instantiating...
Generating Datalog program... [0.000s CPU, 0.001s wall-clock]
Normalizing Datalog program...
Trivial rules: Converted to facts.
Normalizing Datalog program: [0.000s CPU, 0.005s wall-clock]
Preparing model... [0.010s CPU, 0.001s wall-clock]
Generated 16 rules.
Computing model... [0.000s CPU, 0.002s wall-clock]
130 relevant atoms
76 auxiliary atoms
206 